In [0]:

# ================================================================
# NOTEBOOK: nb_silver_product_reviews
# PURPOSE:  Full Refresh Bronze → Silver for Product Reviews
# RUN:      Weekly (after ADF pl_ingest_product_reviews runs)
#           Also run ONCE manually now to seed Silver
# SOURCE:   bronze/product_reviews/ (parquet from ADF/Databricks)
# TARGET:   silver/product_reviews/ (Delta format)

# WHAT THIS NOTEBOOK ADDS BEYOND BRONZE:
#   1. Cleans and validates Rating (must be 1-5)
#   2. Adds SentimentFlag (POSITIVE/NEUTRAL/NEGATIVE from Rating)
#   3. Adds IsVerifiedBool (string "TRUE" → actual boolean)
#   4. Calculates HelpfulnessScore (normalized 0-1)
#   5. Adds ReviewLength (word count proxy for detail quality)
#   6. Adds ReviewYear/Month for trend analysis
#   7. Flags suspicious reviews (very short + extreme rating)
# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import (col, trim, round, when, upper, to_date, current_timestamp, current_date, size, split, lit)
import builtins

STORAGE = "abfss://source@stshopsensedevhj.dfs.core.windows.net"
BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/productreviews/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/productreviews/"


# ── READ Bronze ───────────────────────────────────────────────

bronze_df = spark.read.parquet(BRONZE_PATH)
bronze_df.printSchema()

# ── STEP 1: Remove nulls on key columns ──────────────────
# ReviewID, ProductID, CustomerID, Rating are all important
# Without Rating → cannot calculate product sentiment

bronze_df = (
    bronze_df
    .filter(col("ReviewID").isNotNull())
    .filter(col("ProductID").isNotNull())
    .filter(col("CustomerID").isNotNull())
    .filter(col("Rating").isNotNull())
)

print(f"[CLEAN] After null removal: {bronze_df.count()} rows")

# ── STEP 2: Validate Rating range (must be 1 to 5) ───────────
# Invalid ratings break all sentiment and avg calculations

invalid_ratings = bronze_df.filter(
    (col("Rating").cast("integer") < 1) | (col("Rating").cast("integer") > 5)
).count()

#Check whether invalid rows exist

if invalid_ratings > 0:
    print(f"[WARN] {invalid_ratings} rows have invalid Rating — quarantining")

#Send invalid rows to quarantine

bronze_df.filter(
    (col("Rating").cast("integer") < 1) | (col("Rating").cast("integer") > 5)
    ).write.format("parquet").mode("append") \
.save(f"{STORAGE}/bronze/quarantine/productreviews/")


# Keep only valid ratings
bronze_df = (
    bronze_df
    .filter((col("Rating").cast("integer") >=1)& 
            (col("Rating").cast("integer") <=5))
)
print(f"[VALID] After rating validation: {bronze_df.count()} rows")

# ── STEP 3: Deduplication on ReviewID ────────────────────────
before = bronze_df.count()
bronze_df = bronze_df.dropDuplicates(["ReviewID"])
after = bronze_df.count()
if before != after:
    print(f"[DEDUP] Removed {before - after} duplicate ReviewIDs")

# ── STEP 4: Type casting ──────────────────────────────────────

silver_df = (
    bronze_df
    .withColumn("Rating",   col("Rating").cast("integer"))
    .withColumn("HelpfulVotes",   col("HelpfulVotes").cast("integer"))
    .withColumn("ReviewDate",   to_date("ReviewDate"))
    .withColumn("LastModifiedDate",   to_date("LastModifiedDate"))
)

# ── STEP 5: Standardize string columns ───────────────────────

silver_df = (
    silver_df
    .withColumn("ReviewID",    upper(trim(col("ReviewID"))))
    .withColumn("ProductID",      upper(trim(col("ProductID"))))
    .withColumn("CustomerID",  upper(trim(col("CustomerID"))))
    .withColumn("ReviewTitle",     F.initcap(trim(col("ReviewTitle"))))
    .withColumn("IsVerifiedPurchase",    upper(trim(col("IsVerifiedPurchase"))))
    .withColumn("Source",     upper(trim(col("Source"))))
)

# ── STEP 6: Business derived columns ─────────────────────────

silver_df = (
    silver_df
#Boolean version of IsVerifiedPurchase
    .withColumn("IsVerifiedBool",    col("IsVerifiedPurchase") == "TRUE")

# ── SENTIMENT FLAG ────────────────────────────────────────
    .withColumn("SentimentFlag",  when(col("Rating") > 4, "POSITIVE")
                .when(col("Rating") ==3, "NEUTRAL")
                .otherwise("NEGATIVE"))
    
# ── REVIEW LENGTH ─────────────────────────────────────────
# Word count of the review text (longer = more detailed = more useful)\
    .withColumn("ReviewWordCount",    when(col("ReviewText").isNotNull(),
                size(split(trim(col("ReviewText"))," ")))
                .otherwise(0.0))

#Detailed review flag
    .withColumn("IsDetailedReview",  col("ReviewWordCount") > 10)
    
    # Helpfulness score
    .withColumn(
        "HelpfulnessScore",
        when(
            col("HelpfulVotes").isNotNull(),
            round(col("HelpfulVotes").cast("double") / lit(100), 4)
        ).otherwise(0.0)
    )
    
# Suspicious review flag
    .withColumn("IsSuspicious",
                (col("ReviewWordCount") < 3) &
                (col("Rating").isin(1,5)) &
                (col("IsVerifiedBool") == False))

#   Review time dimensions

    .withColumn("ReviewYear",    F.year("ReviewDate"))
    .withColumn("ReviewMonth",     F.month("ReviewDate"))
    .withColumn("ReviewQuarter",     F.quarter("ReviewDate"))


# Days since review was posted (recency)
    .withColumn("DaysSinceReview",    F.datediff(current_timestamp(),col("ReviewDate")))
                

       
# ── COMBINED QUALITY SCORE ────────────────────────────────
# Is this a recent review? (last 90 days)        
    .withColumn("IsRecentReview",    col("DaysSinceReview") < 90)

    .withColumn("IsHighQualityReview",
                (col("IsVerifiedBool") == True) &
                (col("IsDetailedReview") == True) &
                (col("IsSuspicious") == False))

  
# Metadata
    .withColumn("_silver_load_ts",  F.current_timestamp())
    .withColumn("source",     lit("full_refresh_weekly"))

)


# ── STEP 7: Write Silver as Delta ─────────────────────────────
(
silver_df.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.partitionBy("ReviewYear",  "ReviewMonth")
.save(SILVER_PATH)
)


 # ── STEP 8: Verify + Show Key Metrics ────────────────────────   

total= silver_df.count()
positive = silver_df.filter(col("SentimentFlag") == "SentimentFlag").count()
neutral = silver_df.filter(col("SentimentFlag") == "NEUTRAL").count()
negative = silver_df.filter(col("SentimentFlag") == "NEGATIVE").count()
verifed = silver_df.filter(col("IsVerifiedBool") == True).count()
suspicious = silver_df.filter(col("IsSuspicious") == True).count()
high_qual = silver_df.filter(col("IsHighQualityReview") == True).count()
avg_rating = silver_df.agg(F.avg("Rating")).collect()[0][0]


print(f"\n[DONE] silver/product_reviews/ written: {total} rows")
print(f"\n[SENTIMENT BREAKDOWN]")
print(f"POSITIVE (4-5 stars): {positive} ({builtins.round(positive/total*100, 1)}%)")
print(f" NEUTRAL (3-star): {neutral} ({builtins.round(neutral/total*100, 1)}%)")
print(f"NEGATIVE (1-2 stars): {negative} ({builtins.round(negative/total*100, 1)}%)")


print(f"\n[QUALITY METRICS]")
print(f" Verified purchases: ({builtins.round(verifed/total * 100,1)}%")
print(f"Suspicious reviews:{suspicious}")
print(f" High_quality: {high_qual}")
print(f" Average rating:{builtins.round(avg_rating, 2)} / 5.0")


print("\n[RATING DISTRIBUTION]")
silver_df.groupBy("Rating") \
.agg(F.count("ReviewID").alias("review_count")) \
.orderBy("Rating", ascending=False) \
.show()

print("[TOP REVIEWED PRODUCTS]")
# Which products received the most reviews?

silver_df.groupBy("ProductID") \
.agg( F.count("ReviewID").alias("review_count"),
F.avg("Rating").alias("avg_rating"),
F.sum(when(col("SentimentFlag") == "POSITIVE",1).otherwise(0)).alias("positive_count")
) \
    .withColumn("avg_rating",  round(col("avg_rating"),2)) \
    .orderBy("review_count" , ascending = False) \
    .limit(10) \
    .show() \

display(silver_df.select("ReviewID", "ProductID", "CustomerID",
    "Rating", "SentimentFlag", "ReviewWordCount",
    "IsVerifiedBool", "IsDetailedReview",
    "HelpfulnessScore", "IsSuspicious",
    "IsHighQualityReview", "ReviewDate").limit(10))



root
 |-- ReviewID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- ReviewTitle: string (nullable = true)
 |-- ReviewText: string (nullable = true)
 |-- ReviewDate: date (nullable = true)
 |-- IsVerifiedPurchase: string (nullable = true)
 |-- HelpfulVotes: integer (nullable = true)
 |-- Source: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

[CLEAN] After null removal: 1500 rows
[VALID] After rating validation: 1500 rows

[DONE] silver/product_reviews/ written: 1500 rows

[SENTIMENT BREAKDOWN]
POSITIVE (4-5 stars): 0 (0.0%)
 NEUTRAL (3-star): 216 (14.4%)
NEGATIVE (1-2 stars): 731 (48.7%)

[QUALITY METRICS]
 Verified purchases: (73.9%
Suspicious reviews:0
 High_quality: 0
 Average rating:3.9 / 5.0

[RATING DISTRIBUTION]
+------+------------+
|Rating|review_count|
+------+------------+
|     5|         553|
|     4|         526|
|     3|         216|
|     

ReviewID,ProductID,CustomerID,Rating,SentimentFlag,ReviewWordCount,IsVerifiedBool,IsDetailedReview,HelpfulnessScore,IsSuspicious,IsHighQualityReview,ReviewDate
REV0000001,PROD0249,CUST00087,5,POSITIVE,4.0,true,false,0.09,false,false,2024-05-10
REV0000002,PROD0135,CUST00309,5,POSITIVE,5.0,false,false,0.66,false,false,2024-05-01
REV0000003,PROD0226,CUST00336,4,NEGATIVE,4.0,true,false,1.3,false,false,2024-02-15
REV0000004,PROD0071,CUST00080,5,POSITIVE,4.0,false,false,1.27,false,false,2024-02-04
REV0000005,PROD0063,CUST00252,3,NEUTRAL,4.0,true,false,0.49,false,false,2024-04-29
REV0000006,PROD0259,CUST00014,3,NEUTRAL,5.0,false,false,1.42,false,false,2024-03-09
REV0000007,PROD0074,CUST00425,4,NEGATIVE,4.0,false,false,0.2,false,false,2024-06-04
REV0000008,PROD0137,CUST00210,4,NEGATIVE,5.0,true,false,0.5,false,false,2024-03-26
REV0000009,PROD0016,CUST00205,5,POSITIVE,4.0,true,false,0.42,false,false,2024-01-13
REV0000010,PROD0251,CUST00419,5,POSITIVE,5.0,true,false,1.25,false,false,2024-01-30


In [0]:
display(silver_df)

ReviewID,ProductID,CustomerID,Rating,ReviewTitle,ReviewText,ReviewDate,IsVerifiedPurchase,HelpfulVotes,source,LastModifiedDate,IsVerifiedBool,SentimentFlag,ReviewWordCount,IsDetailedReview,HelpfulnessScore,IsSuspicious,ReviewYear,ReviewMonth,ReviewQuarter,DaysSinceReview,IsRecentReview,IsHighQualityReview,_silver_load_ts
REV0000001,PROD0249,CUST00087,5,Rating: 5/5,"Excellent value, very satisfied.",2024-05-10,TRUE,9,full_refresh_weekly,2024-05-10,true,POSITIVE,4.0,false,0.09,false,2024,5,2,795,false,false,2026-07-14T16:21:30.411774Z
REV0000002,PROD0135,CUST00309,5,Rating: 5/5,"Exactly as expected, good purchase.",2024-05-01,FALSE,66,full_refresh_weekly,2024-05-01,false,POSITIVE,5.0,false,0.66,false,2024,5,2,804,false,false,2026-07-14T16:21:30.411774Z
REV0000003,PROD0226,CUST00336,4,Rating: 4/5,"Great product, highly recommend!",2024-02-15,TRUE,130,full_refresh_weekly,2024-02-15,true,NEGATIVE,4.0,false,1.3,false,2024,2,1,880,false,false,2026-07-14T16:21:30.411774Z
REV0000004,PROD0071,CUST00080,5,Rating: 5/5,"Not as described, disappointed.",2024-02-04,FALSE,127,full_refresh_weekly,2024-02-04,false,POSITIVE,4.0,false,1.27,false,2024,2,1,891,false,false,2026-07-14T16:21:30.411774Z
REV0000005,PROD0063,CUST00252,3,Rating: 3/5,"Excellent value, very satisfied.",2024-04-29,TRUE,49,full_refresh_weekly,2024-04-29,true,NEUTRAL,4.0,false,0.49,false,2024,4,2,806,false,false,2026-07-14T16:21:30.411774Z
REV0000006,PROD0259,CUST00014,3,Rating: 3/5,"Exactly as expected, good purchase.",2024-03-09,FALSE,142,full_refresh_weekly,2024-03-09,false,NEUTRAL,5.0,false,1.42,false,2024,3,1,857,false,false,2026-07-14T16:21:30.411774Z
REV0000007,PROD0074,CUST00425,4,Rating: 4/5,"Not as described, disappointed.",2024-06-04,FALSE,20,full_refresh_weekly,2024-06-04,false,NEGATIVE,4.0,false,0.2,false,2024,6,2,770,false,false,2026-07-14T16:21:30.411774Z
REV0000008,PROD0137,CUST00210,4,Rating: 4/5,Average quality for the price.,2024-03-26,TRUE,50,full_refresh_weekly,2024-03-26,true,NEGATIVE,5.0,false,0.5,false,2024,3,1,840,false,false,2026-07-14T16:21:30.411774Z
REV0000009,PROD0016,CUST00205,5,Rating: 5/5,"Not as described, disappointed.",2024-01-13,TRUE,42,full_refresh_weekly,2024-01-13,true,POSITIVE,4.0,false,0.42,false,2024,1,1,913,false,false,2026-07-14T16:21:30.411774Z
REV0000010,PROD0251,CUST00419,5,Rating: 5/5,"Exactly as expected, good purchase.",2024-01-30,TRUE,125,full_refresh_weekly,2024-01-30,true,POSITIVE,5.0,false,1.25,false,2024,1,1,896,false,false,2026-07-14T16:21:30.411774Z
